In [1]:
import pandas as pd
from typing import Tuple
import math
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Cleaning Schedule
This notebook provides a clean version of the raw given schedule {schedule_name}.csv file.
If does the following:
1. Rename duplicate columns.
2. Renaming any instances of 'ATH' to 'OAK', i.e. rebranding the Sacramento Athletics to Oakland Athletics.
3. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.
4. Adding 'homedistancetraveled' and 'visdistancetraveled' columns, which contains the distance in miles traveled from the team's previous games.
5. Adding 'homerestdays' and 'visrestdays' columns, which contains the number of days between the current game and the previous game for the home and visiting teams.
6. Save to a new csv, `.data/clean/{schedule_name}_cleaned.csv`

In [2]:
pd.set_option('display.max_columns', None)

In [ ]:
schedule_name = "2025schedule"

schedule = pd.read_csv(Path.cwd() / "analysis" / "data" / "raw" / f"{schedule_name}.csv")
schedule.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/data/raw/2024schedule.csv'

## 1. Rename Duplicate Columns

In [ ]:
schedule = schedule.rename(columns={'League':'VisitorLeague', 'Game':'VisitorGame', 'League.1':'HomeLeague', 'Game.1':'HomeGame'})
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Postponed,Makeup
0,20230330,0,Thu,BAL,AL,1,BOS,AL,1,d,NaN,NaN
1,20230330,0,Thu,MIL,NL,1,CHN,NL,1,d,NaN,NaN
2,20230330,0,Thu,PIT,NL,1,CIN,NL,1,d,NaN,NaN
3,20230330,0,Thu,CHA,AL,1,HOU,AL,1,n,NaN,NaN
4,20230330,0,Thu,MIN,AL,1,KCA,AL,1,d,NaN,NaN


## 2. Renaming 'ATH' team names to 'OAK'

In [ ]:
schedule['Visitor'] = schedule['Visitor'].apply(lambda x: 'OAK' if x == 'ATH' else x)
schedule['Home'] = schedule['Home'].apply(lambda x: 'OAK' if x == 'ATH' else x)


## 3. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.

In [ ]:
def get_timestamp(date: int) -> pd.Timestamp:
    """For a given date int of format YYYYMMDD, gets its game start timestamp."""

    date = str(date)

    y = int(date[:4])
    m = int(date[4:6])
    d = int(date[6:])
    return pd.Timestamp(year=y, month=m, day=d)

# Add the timestamp column

schedule['timestamp'] = schedule['Date'].apply(get_timestamp)
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Postponed,Makeup,timestamp
0,20230330,0,Thu,BAL,AL,1,BOS,AL,1,d,NaN,NaN,2023-03-30
1,20230330,0,Thu,MIL,NL,1,CHN,NL,1,d,NaN,NaN,2023-03-30
2,20230330,0,Thu,PIT,NL,1,CIN,NL,1,d,NaN,NaN,2023-03-30
3,20230330,0,Thu,CHA,AL,1,HOU,AL,1,n,NaN,NaN,2023-03-30
4,20230330,0,Thu,MIN,AL,1,KCA,AL,1,d,NaN,NaN,2023-03-30


## 4. Adding 'homedistancetraveled' and 'visdistancetraveled' Columns

In [ ]:
# Add temporary latitude and longitude columns
parks = pd.read_csv('Parks.csv')
schedule = pd.merge(schedule, parks[['PARKID', 'Latitude', 'Longitude']], how='left', left_on='Location', right_on='PARKID').reset_index(drop=True)
schedule = schedule.drop('PARKID', axis=1)

KeyError: 'Location'

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Returns Haversine distance between two pairs of latitudes and longitudes."""
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [ ]:
def get_closest_home_game_coords(team: str, idx: int) -> Tuple[float, float]:
    """For the given team, returns the latitude and longitude of the closest home game they played
    to wherever they are in schedule, given by idx. If no home game is found, it returns None, None."""
    
    n_games = len(schedule)
    
    before = idx - 1
    after = idx + 1
    while before >= 0 or after <= n_games - 1:
        if before >= 0:
            before_gm = schedule.iloc[before]
            if before_gm['Home'] == team:
                return before_gm['Latitude'], before_gm['Longitude']
            
            before -= 1

        if after <= n_games - 1:
            after_gm = schedule.iloc[after]
            if after_gm['Home'] == team:
                return after_gm['Latitude'], after_gm['Longitude']
            
            after += 1
            
    return None, None

In [ ]:
home_dists = []
away_dists = []

last_games = {} # Mapping of team id to (lat, lon, season) tuple of previous game - if empty, just use np.nan

for i, game in tqdm(schedule.iterrows()):
    
    cur_lat = game['Latitude']
    cur_lon = game['Longitude']
    
    home_team = game['Home']
    away_team = game['Visitor']
    
    for team, dists in zip([home_team, away_team], [home_dists, away_dists]):
        if team in last_games:
            last_lat, last_lon = last_games[team]           
            
        else: # Never played a game before
            last_lat, last_lon = get_closest_home_game_coords(team, i)
            
        if last_lat is not None:
            dist = haversine(last_lat, last_lon, cur_lat, cur_lon)
            dists.append(dist)
        else:
            dists.append(np.nan)
            
        last_games[team] = (cur_lat, cur_lon)

schedule['homedistancetraveled'] = home_dists    
schedule['visdistancetraveled'] = away_dists
schedule.head()

2430it [00:00, 30021.53it/s]


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled
0,20240320,0,Wednesday,LAN,NL,1,SDN,NL,1,n,SEO01,NaN,NaN,2024-03-20,37.497700,128.867100,5990.094728,0.000000
1,20240321,0,Thursday,SDN,NL,2,LAN,NL,2,n,SEO01,NaN,NaN,2024-03-21,37.497700,128.867100,0.000000,0.000000
2,20240328,0,Thursday,MIL,NL,1,NYN,NL,1,d,NYC20,Rain,20240329,2024-03-28,40.757134,-73.845840,0.000000,742.294029
3,20240328,0,Thursday,ANA,AL,1,BAL,AL,1,d,BAL12,NaN,NaN,2024-03-28,39.283944,-76.621572,0.000000,2301.420160
4,20240328,0,Thursday,ATL,NL,1,PHI,NL,1,d,PHI13,Rain,20240329,2024-03-28,39.906109,-75.166485,0.000000,660.304769


## 5. Add Rest Day Columns

In [ ]:
home_rest_days = []
away_rest_days = []

last_played = {}

for _, game in tqdm(schedule.iterrows()):
    home_team = game['Home']
    away_team = game['Visitor']
    timestamp = game['timestamp']
    
    prev_home_t = last_played.get(home_team)
    prev_away_t = last_played.get(away_team)
    
    home_rest_days.append((timestamp.floor('D') - prev_home_t.floor('D')).days if prev_home_t is not None else np.nan)
    away_rest_days.append((timestamp.floor('D') - prev_away_t.floor('D')).days if prev_away_t is not None else np.nan)

    last_played[home_team] = timestamp
    last_played[away_team] = timestamp
 
schedule['homerestdays'] = home_rest_days
schedule['visrestdays'] = away_rest_days
schedule.head()

2430it [00:00, 10853.32it/s]


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled,homerestdays,visrestdays
0,20240320,0,Wednesday,LAN,NL,1,SDN,NL,1,n,SEO01,NaN,NaN,2024-03-20,37.497700,128.867100,5990.094728,0.000000,NaN,NaN
1,20240321,0,Thursday,SDN,NL,2,LAN,NL,2,n,SEO01,NaN,NaN,2024-03-21,37.497700,128.867100,0.000000,0.000000,1.0,1.0
2,20240328,0,Thursday,MIL,NL,1,NYN,NL,1,d,NYC20,Rain,20240329,2024-03-28,40.757134,-73.845840,0.000000,742.294029,NaN,NaN
3,20240328,0,Thursday,ANA,AL,1,BAL,AL,1,d,BAL12,NaN,NaN,2024-03-28,39.283944,-76.621572,0.000000,2301.420160,NaN,NaN
4,20240328,0,Thursday,ATL,NL,1,PHI,NL,1,d,PHI13,Rain,20240329,2024-03-28,39.906109,-75.166485,0.000000,660.304769,NaN,NaN


## 6. Save to `.csv`

In [ ]:
schedule.to_csv(Path.cwd() / "analysis" / "data" / "clean" / f"{schedule_name}_clean.csv", index=False)